<h3 style="color:#6FA8DC; font-weight:bold">Multivariate Imputation → KNN Imputer</h3>

### What is Multivariate Imputation?
Instead of filling a missing value using only its own column, we use **other features of the same row** to estimate it.

**KNN Imputation** finds similar rows (nearest neighbors) and uses their values to fill the missing value.

```text
Missing value
     ↓
Find similar rows
     ↓
Take their known values
     ↓
Calculate weighted/average value
     ↓
Imputed value
```


### Why use KNN Imputer?
Mean/median imputation ignores relationships between features. KNN can use relationships such as **Age, Pclass and Fare** together.

**When useful**
- Numerical features have meaningful relationships.
- Dataset is not extremely large.
- Similar observations tend to have similar values.

**Advantages**
- Uses multiple features.
- Can preserve relationships better than simple mean/median in suitable data.
- `weights='distance'` gives more influence to closer observations.

**Disadvantages**
- Computationally more expensive.
- Sensitive to feature scale.
- Usually unsuitable for categorical variables.
- Choice of `n_neighbors` affects results.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv('train(1).csv')[['Age','Pclass','Fare','Survived']]
df.head()

In [ ]:
df.isnull().mean() * 100

### Important → KNN is distance-based
KNN uses distances. Therefore, features with very different scales can dominate the distance. **In modern ML, scaling should be part of the Pipeline before KNN.**

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('imputer', KNNImputer(n_neighbors=3, weights='distance')),
    ('model', LogisticRegression(max_iter=1000))
])

knn_pipe.fit(X_train, y_train)
y_pred = knn_pipe.predict(X_test)

accuracy_score(y_test, y_pred)

### Why Pipeline?
The scaler and imputer are **fit only on training data**. Test data is transformed using the learned training information.

This prevents **data leakage** and keeps production preprocessing consistent.

### KNN parameters
- `n_neighbors` → number of nearby rows considered.
- `weights='uniform'` → all neighbors contribute equally.
- `weights='distance'` → closer neighbors contribute more.
- `metric` → distance metric used by KNN.

### Revision
**KNN Imputer = missing value estimated from similar rows using multiple numerical features.**